Script: this piece of code runs the SEOF package on multiple file input for 500mb geopotential height in the PNA and NAt regions

Notes: needs to be run individually for each model - check the boxes that have a *change me* tag on the top. 

Output files: SEOFs for North Pacific & North Atlantic

PNA & NAT

SEOF, SPCS, FVAR, TVAR

For each model the z500 fields with and without the ensemble mean removed. 
_zgDJFerem
_zgDJF'

In [2]:
#load packages needed in this notebook
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os
from eofs.standard import Eof
import regionmask
import cartopy.crs as ccrs
from natsort import natsorted 


In [3]:
#set up the data directory and load in lon + lat + time dimensions
outputdir='/glade/campaign/cgd/cas/nmaher/canesm5_lens/Amon/zg/' 
outputdir2='/glade/work/nmaher/SEOF_output/'
model='CanESM5-SSP585'


#option for merging files
ds_fx1 = xr.open_dataset(outputdir+'zg_Amon_CanESM5_historical_r10i1p2f1_g025.nc')
ds_fx2 = xr.open_dataset(outputdir+'zg_Amon_CanESM5_ssp585_r10i1p2f1_g025.nc')


ds_fx = xr.merge([ds_fx1, ds_fx2])

lon = ds_fx.lon
lat = ds_fx.lat
time = ds_fx.time

In [4]:
#*change me*
#set up list of files for each member

files = natsorted(os.listdir(outputdir))
files = [s for s in files if "historical" in s and "i1p2f1" in s]

files2 = natsorted(os.listdir(outputdir))
files2 = [s for s in files2 if "ssp585" in s and "i1p2f1" in s]

n = len(files)#10 #25 for canesm5 #len(files) #10 for ipsl 10 for mirocesl
ne = np.empty(n)
for ii in range(n):
        ne[ii] = ii

        


In [5]:
#loop through ensemble members and load data
zg_all = np.empty((n,len(time),len(lat),len(lon)))
zg_all = xr.DataArray(zg_all, coords=[ne, time, lat, lon], dims=["member", "time", "lat", "lon"])

for ii in range(n):
        #print(ii)
        filenameH = outputdir+files[ii]
        filenameS = outputdir+files2[ii]
        ds_memberH = xr.open_dataset(filenameH)
        ds_memberS = xr.open_dataset(filenameS)
        ds_member=xr.merge([ds_memberH, ds_memberS])
        zg = ds_member.zg
        zg_all[ii,:,:,:] = zg.values


In [6]:
zg_all

<xarray.DataArray (member: 25, time: 3012, lat: 72, lon: 144)>
array([[[[5056.75195312, 5057.12890625, 5057.49365234, ...,
          5055.5546875 , 5055.96484375, 5056.36376953],
         [5065.56640625, 5065.54638672, 5065.50097656, ...,
          5065.23974609, 5065.43603516, 5065.53759766],
         [5070.69824219, 5069.81445312, 5069.02050781, ...,
          5073.18164062, 5072.46289062, 5071.60693359],
         ...,
         [5160.81835938, 5161.77099609, 5162.46923828, ...,
          5156.56640625, 5158.1953125 , 5159.62060547],
         [5130.82910156, 5131.74951172, 5132.55419922, ...,
          5127.41162109, 5128.65478516, 5129.796875  ],
         [5111.14550781, 5111.78417969, 5112.39160156, ...,
          5109.04541016, 5109.77587891, 5110.47607422]],

        [[5019.75585938, 5019.76953125, 5019.79101562, ...,
          5019.76074219, 5019.75097656, 5019.75      ],
         [5020.52392578, 5020.03076172, 5019.52783203, ...,
          5021.70654297, 5021.38378906, 5020.98291016],
         [5021.0859375 , 5019.97460938, 5018.96142578, ...,
          5024.41064453, 5023.37353516, 5022.24121094],
...
         [5283.44677734, 5282.99267578, 5282.53710938, ...,
          5284.91796875, 5284.39404297, 5283.90917969],
         [5283.67871094, 5283.46044922, 5283.22265625, ...,
          5284.22949219, 5284.05957031, 5283.87744141],
         [5281.81787109, 5281.78125   , 5281.72753906, ...,
          5281.81591797, 5281.8359375 , 5281.83642578]],

        [[5217.10400391, 5218.10742188, 5219.08789062, ...,
          5213.96484375, 5215.03076172, 5216.07763672],
         [5224.37402344, 5225.81054688, 5227.19189453, ...,
          5219.56787109, 5221.27001953, 5222.86669922],
         [5225.42333984, 5227.25      , 5229.08447266, ...,
          5219.50976562, 5221.61230469, 5223.56298828],
         ...,
         [5159.95849609, 5159.37988281, 5158.84472656, ...,
          5162.01123047, 5161.26513672, 5160.58447266],
         [5157.86865234, 5157.59619141, 5157.35400391, ...,
          5158.89160156, 5158.51318359, 5158.17382812],
         [5162.39404297, 5162.16552734, 5161.95263672, ...,
          5163.17041016, 5162.89697266, 5162.63818359]]]])
Coordinates:
  * member   (member) float64 0.0 1.0 2.0 3.0 4.0 ... 20.0 21.0 22.0 23.0 24.0
  * time     (time) object 1850-01-16 12:00:00 ... 2100-12-16 12:00:00
  * lat      (lat) float64 -88.75 -86.25 -83.75 -81.25 ... 83.75 86.25 88.75
  * lon      (lon) float64 1.25 3.75 6.25 8.75 11.25 ... 351.2 353.8 356.2 358.8

In [7]:
#select season
zg_DJF_full = zg_all.where(zg_all['time.season'] == 'DJF')

In [8]:
zg_DJF_full

<xarray.DataArray (member: 25, time: 3012, lat: 72, lon: 144)>
array([[[[5056.75195312, 5057.12890625, 5057.49365234, ...,
          5055.5546875 , 5055.96484375, 5056.36376953],
         [5065.56640625, 5065.54638672, 5065.50097656, ...,
          5065.23974609, 5065.43603516, 5065.53759766],
         [5070.69824219, 5069.81445312, 5069.02050781, ...,
          5073.18164062, 5072.46289062, 5071.60693359],
         ...,
         [5160.81835938, 5161.77099609, 5162.46923828, ...,
          5156.56640625, 5158.1953125 , 5159.62060547],
         [5130.82910156, 5131.74951172, 5132.55419922, ...,
          5127.41162109, 5128.65478516, 5129.796875  ],
         [5111.14550781, 5111.78417969, 5112.39160156, ...,
          5109.04541016, 5109.77587891, 5110.47607422]],

        [[5019.75585938, 5019.76953125, 5019.79101562, ...,
          5019.76074219, 5019.75097656, 5019.75      ],
         [5020.52392578, 5020.03076172, 5019.52783203, ...,
          5021.70654297, 5021.38378906, 5020.98291016],
         [5021.0859375 , 5019.97460938, 5018.96142578, ...,
          5024.41064453, 5023.37353516, 5022.24121094],
...
         [          nan,           nan,           nan, ...,
                    nan,           nan,           nan],
         [          nan,           nan,           nan, ...,
                    nan,           nan,           nan],
         [          nan,           nan,           nan, ...,
                    nan,           nan,           nan]],

        [[5217.10400391, 5218.10742188, 5219.08789062, ...,
          5213.96484375, 5215.03076172, 5216.07763672],
         [5224.37402344, 5225.81054688, 5227.19189453, ...,
          5219.56787109, 5221.27001953, 5222.86669922],
         [5225.42333984, 5227.25      , 5229.08447266, ...,
          5219.50976562, 5221.61230469, 5223.56298828],
         ...,
         [5159.95849609, 5159.37988281, 5158.84472656, ...,
          5162.01123047, 5161.26513672, 5160.58447266],
         [5157.86865234, 5157.59619141, 5157.35400391, ...,
          5158.89160156, 5158.51318359, 5158.17382812],
         [5162.39404297, 5162.16552734, 5161.95263672, ...,
          5163.17041016, 5162.89697266, 5162.63818359]]]])
Coordinates:
  * member   (member) float64 0.0 1.0 2.0 3.0 4.0 ... 20.0 21.0 22.0 23.0 24.0
  * time     (time) object 1850-01-16 12:00:00 ... 2100-12-16 12:00:00
  * lat      (lat) float64 -88.75 -86.25 -83.75 -81.25 ... 83.75 86.25 88.75
  * lon      (lon) float64 1.25 3.75 6.25 8.75 11.25 ... 351.2 353.8 356.2 358.8

In [9]:
#take seasonal mean for masked and full regions
zg_DJF_full = zg_DJF_full.rolling(min_periods=3, center=True, time=3).mean()

# make annual mean
zg_DJF_full = zg_DJF_full.groupby('time.year').mean('time')

In [10]:
#set up the input removing the first year as JF not DJF - take anomalies relative to ensemble mean

zg_DJF_2_full=zg_DJF_full[:,1:,:,:]
zg_DJF_2_full=zg_DJF_2_full.values
zg_DJF_2e_full = zg_DJF_2_full - np.ma.average(zg_DJF_2_full,axis=0)[np.newaxis,:,:,:]

In [11]:
#mask the PNA region
lat_range=[20,90]
lon_range=[110,260]

lats=lat.values
lons=lon.values

ilat=np.logical_or(lats<20,lats>90)
ilon = np.logical_or(lons<110,lons>260)

ilat2d=np.broadcast_to(ilat[:,np.newaxis], zg_DJF_2_full[0,0,...].shape)
ilon2d=np.broadcast_to(ilon[np.newaxis,:], zg_DJF_2_full[0,0,...].shape)

mask2d=np.logical_or(ilat2d,ilon2d)
mask4d=np.broadcast_to(mask2d[np.newaxis,np.newaxis,:,:],zg_DJF_2_full.shape)

masked_zgPNA=np.ma.masked_array(zg_DJF_2_full,mask=mask4d)

In [12]:
#set up time dimension to loop through
ntt=len(zg_DJF_2e_full[1,:,1,1])-4
esize=len(zg_DJF_2e_full[:,1,1,1])


In [13]:
#do the SEOF calculation

#weight with cosine of lat
coslat = np.cos(np.deg2rad(lat))

#how many eofs are we doing?
neof=3

#set up output dimensions
pna_seof = np.ma.masked_equal(np.zeros([ntt,neof,72,144]),0)
pna_spcs = np.ma.masked_equal(np.zeros([ntt,esize*5,neof]),0)
#pna_pcs = np.ma.masked_equal(np.zeros([ntt,200,neof]),0)

pna_seof_fracVarExp_arr = np.ma.masked_equal(np.zeros([ntt,neof]),0)
pna_seof_totalVar_arr = np.ma.masked_equal(np.zeros([ntt,1]),0)
pna_eofs_northTest = np.ma.masked_equal(np.zeros([ntt,neof]),0)


#loop through each year and do the EOFs
for i in range(ntt):

    zg_DJF_3 = masked_zgPNA[:,i:i+5,...].reshape(-1,72,144)

    
   
    a=np.sqrt(coslat)
    a2=a.values
    wgts = np.broadcast_to(a2[np.newaxis,:,np.newaxis], zg_DJF_3.shape)

    solver = Eof(zg_DJF_3, weights=wgts)


    eofs = solver.eofs(neofs=neof, eofscaling=2)
    pcs = solver.pcs(npcs=neof,pcscaling=1)
    fracvar = solver.varianceFraction(neigs=neof)
    total_variance = solver.totalAnomalyVariance()
    
    pna_eofs_northTest[i,:] = solver.northTest(neigs=neof, vfscaled=True)
    
    spcs = np.ma.masked_array(np.zeros(pcs.shape),0)
    for j in range(len(pcs[0,:])):
        spcs[:,j] = (pcs[:,j] - np.mean(pcs[:,j]))/np.std(pcs[:,j])



    pna_seof[i,...] = eofs
    pna_spcs[i,...] = spcs
    pna_seof_fracVarExp_arr[i,...] = fracvar
    pna_seof_totalVar_arr[i,...] = total_variance


In [14]:
#save output
eof_T='_PNA'

np.savez_compressed(outputdir2+model+eof_T+'_SEOF.npz', data=pna_seof.data, mask=pna_seof.mask)
np.savez_compressed(outputdir2+model+eof_T+'_SPCS.npz', data=pna_spcs.data, mask=pna_spcs.mask)
np.savez_compressed(outputdir2+model+eof_T+'_FVAR.npz', data=pna_seof_fracVarExp_arr.data, mask=pna_seof_fracVarExp_arr.mask)
np.savez_compressed(outputdir2+model+eof_T+'_TVAR.npz', data=pna_seof_totalVar_arr.data, mask=pna_seof_totalVar_arr.mask)


In [15]:
#mask the NAt region
lat_range=[20,90]
lon_range=[280,10]

ilat=np.logical_or(lats<20,lats>90)
ilon = np.logical_and(lons>10,lons<280)

ilat2d=np.broadcast_to(ilat[:,np.newaxis], zg_DJF_2_full[0,0,...].shape)
ilon2d=np.broadcast_to(ilon[np.newaxis,:], zg_DJF_2_full[0,0,...].shape)

mask2d=np.logical_or(ilat2d,ilon2d)
mask4d=np.broadcast_to(mask2d[np.newaxis,np.newaxis,:,:],zg_DJF_2_full.shape)

masked_zgNAT=np.ma.masked_array(zg_DJF_2_full,mask=mask4d)

In [16]:
#do the SEOF calculation

#weight with cosine of lat
coslat = np.cos(np.deg2rad(lat))

#how many eofs are we doing?
neof=1

#set up output dimensions
nat_seof = np.ma.masked_equal(np.zeros([ntt,neof,72,144]),0)
nat_spcs = np.ma.masked_equal(np.zeros([ntt,esize*5,neof]),0)
#pna_pcs = np.ma.masked_equal(np.zeros([ntt,200,neof]),0)

nat_seof_fracVarExp_arr = np.ma.masked_equal(np.zeros([ntt,neof]),0)
nat_seof_totalVar_arr = np.ma.masked_equal(np.zeros([ntt,1]),0)
nat_eofs_northTest = np.ma.masked_equal(np.zeros([ntt,neof]),0)


#loop through each year and do the EOFs
for i in range(ntt):

    zg_DJF_3 = masked_zgNAT[:,i:i+5,...].reshape(-1,72,144)

    
   
    a=np.sqrt(coslat)
    a2=a.values
    wgts = np.broadcast_to(a2[np.newaxis,:,np.newaxis], zg_DJF_3.shape)

    solver = Eof(zg_DJF_3, weights=wgts)


    eofs = solver.eofs(neofs=neof, eofscaling=2)
    pcs = solver.pcs(npcs=neof,pcscaling=1)
    fracvar = solver.varianceFraction(neigs=neof)
    total_variance = solver.totalAnomalyVariance()
    
    nat_eofs_northTest[i,:] = solver.northTest(neigs=neof, vfscaled=True)
    
    spcs = np.ma.masked_array(np.zeros(pcs.shape),0)
    for j in range(len(pcs[0,:])):
        spcs[:,j] = (pcs[:,j] - np.mean(pcs[:,j]))/np.std(pcs[:,j])


    nat_seof[i,...] = eofs
    nat_spcs[i,...] = spcs
    nat_seof_fracVarExp_arr[i,...] = fracvar
    nat_seof_totalVar_arr[i,...] = total_variance


In [17]:
#save output
eof_T='_NAT'

np.savez_compressed(outputdir2+model+eof_T+'_SEOF.npz', data=nat_seof.data, mask=nat_seof.mask)
np.savez_compressed(outputdir2+model+eof_T+'_SPCS.npz', data=nat_spcs.data, mask=nat_spcs.mask)
np.savez_compressed(outputdir2+model+eof_T+'_FVAR.npz', data=nat_seof_fracVarExp_arr.data, mask=nat_seof_fracVarExp_arr.mask)
np.savez_compressed(outputdir2+model+eof_T+'_TVAR.npz', data=nat_seof_totalVar_arr.data, mask=nat_seof_totalVar_arr.mask)

In [18]:
#save DJF emeanremoved zg
eof_T='_zgDJFerem'
np.savez_compressed(outputdir2+model+eof_T+'_SEOF.npz', data=zg_DJF_2e_full.data, mask=zg_DJF_2e_full.mask)


eof_T='_zgDJF'

np.savez_compressed(outputdir2+model+eof_T+'_SEOF.npz', data=zg_DJF_2_full.data, mask=zg_DJF_2e_full.mask)
